# Analyse Éthique du Turnover : HumanForYou
## Pipeline complète : EDA → Classification (éthique vs non-éthique)

**Objectif** : Identifier les facteurs organisationnels de départ des employés et proposer des leviers d'action RH, tout en respectant un cadre éthique strict.

**Dataset** : 5 fichiers CSV (general_data, employee_survey, manager_survey, in_time, out_time)

# Cadrage du projet

L'entreprise **HumanForYou** (~4 000 employés, Inde) connaît un taux de rotation annuel de ~15 %. La direction souhaite **identifier les leviers organisationnels** sur lesquels agir pour réduire ce turnover.

**Objectif du notebook** : construire un modèle de classification prédisant l'attrition, puis interpréter les résultats pour formuler des recommandations RH concrètes.

> **Note** : L'analyse éthique détaillée et la bibliographie font l'objet de livrables séparés. Ce notebook se concentre sur la démarche technique.

## Approche comparative : filtre éthique

Pour quantifier l'impact des variables sensibles sur les performances, nous exécutons la **même pipeline** deux fois grâce à un feature flag `ethical_filter` :

| `ethical_filter` | Dataset | Variables retirées |
|---|---|---|
| `True` | `df` | Gender, Age, MaritalStatus, DistanceFromHome, avg/std_work_hours, days_absent |
| `False` | `df_full` | Aucune (toutes conservées) |

Les résultats sont comparés en Partie 3.

# Partie 1 : Analyse Exploratoire des Données (EDA)

## 1.1 Environnement et imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
import time

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Seed pour la reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline

print("Environnement prêt ✓")

## 1.2 Chargement des données

In [ ]:
# Chargement des 5 fichiers CSV
general = pd.read_csv('../data/general_data.csv')
emp_survey = pd.read_csv('../data/employee_survey_data.csv')
mgr_survey = pd.read_csv('../data/manager_survey_data.csv')
in_time = pd.read_csv('../data/in_time.csv')
out_time = pd.read_csv('../data/out_time.csv')

print(f"   general_data: {general.shape}")
print(f"employee_survey: {emp_survey.shape}")
print(f" manager_survey: {mgr_survey.shape}")
print(f"        in_time: {in_time.shape}")
print(f"       out_time: {out_time.shape}")

## 1.3 Exploration initiale

Avant toute transformation, on examine la structure de chaque table : dimensions, types de colonnes, statistiques descriptives et valeurs manquantes. Cela permet de détecter rapidement les anomalies et de comprendre la nature des données.

In [ ]:
# Aperçu structuré de chaque table : dimensions, types, statistiques descriptives, valeurs manquantes
datasets = {
    "general":    general,
    "emp_survey": emp_survey,
    "mgr_survey": mgr_survey,
    "in_time":    in_time,
    "out_time":   out_time,
}

for name, tbl in datasets.items():
    print(f"\n{'='*60}")
    print(f" {name} : {tbl.shape[0]} lignes × {tbl.shape[1]} colonnes")
    print(f"{'='*60}")
    print("\n--- Types de colonnes ---")
    print(tbl.dtypes.value_counts().to_string())
    print(f"\n--- Valeurs manquantes : {tbl.isnull().sum().sum()} ---")
    missing_cols = tbl.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0]
    if len(missing_cols) > 0:
        print(missing_cols.to_string())
    print("\n--- Statistiques descriptives ---")
    display(tbl.describe(include='all').T)

## 1.4 Feature Engineering : Heures de travail

Les fichiers `in_time` et `out_time` contiennent les horodatages bruts d'entrée/sortie pour chaque jour ouvré de 2015. Sous cette forme, les données ne sont pas exploitables par un modèle de ML.

On en extrait donc des **métriques agrégées par employé**, plus informatives :
- **avg_work_hours** : nombre moyen d'heures travaillées par jour
- **std_work_hours** : régularité des horaires (écart-type)
- **days_absent** : nombre de jours d'absence (NaN dans les pointages)

Ces features résument le comportement temporel de chaque employé en 3 chiffres exploitables.

In [ ]:
# Renommer la première colonne (non nommée) en EmployeeID
in_time = in_time.rename(columns={in_time.columns[0]: 'EmployeeID'})
out_time = out_time.rename(columns={out_time.columns[0]: 'EmployeeID'})

# Convertir les colonnes de dates en datetime
date_cols = in_time.columns[1:]

in_time_dt = in_time.copy()
out_time_dt = out_time.copy()
for col in date_cols:
    in_time_dt[col] = pd.to_datetime(in_time_dt[col], errors='coerce')
    out_time_dt[col] = pd.to_datetime(out_time_dt[col], errors='coerce')

# Calcul des heures travaillées par jour
work_hours = pd.DataFrame()
work_hours['EmployeeID'] = in_time_dt['EmployeeID']

for col in date_cols:
    diff = (out_time_dt[col] - in_time_dt[col]).dt.total_seconds() / 3600
    work_hours[col] = diff

# Agrégation par employé
time_features = pd.DataFrame()
time_features['EmployeeID'] = work_hours['EmployeeID']
time_features['avg_work_hours'] = work_hours[date_cols].mean(axis=1)
time_features['std_work_hours'] = work_hours[date_cols].std(axis=1)
time_features['days_absent'] = work_hours[date_cols].isnull().sum(axis=1)

print(time_features.describe())
time_features.head()

## 1.5 Fusion des datasets

On fusionne toutes les tables sur `EmployeeID` (jointure à gauche) pour obtenir un dataset unique. C'est indispensable pour que chaque ligne représente un employé avec toutes ses caractéristiques.

In [ ]:
# Fusion progressive sur EmployeeID
df = general.merge(emp_survey, on='EmployeeID', how='left')
df = df.merge(mgr_survey, on='EmployeeID', how='left')
df = df.merge(time_features, on='EmployeeID', how='left')

print(f"Dataset fusionné : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head()

## 1.6 Analyse des valeurs manquantes

Les valeurs manquantes peuvent biaiser les analyses et empêcher l'entraînement de certains modèles. On visualise leur répartition pour choisir la stratégie d'imputation adaptée (section 1.7).

In [ ]:
# Matrice de valeurs manquantes
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

msno.matrix(df, ax=axes[0], sparkline=False, fontsize=8)
axes[0].set_title('Matrice des valeurs manquantes', fontsize=14)

# Barplot des valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    missing.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('Colonnes avec valeurs manquantes')
    axes[1].set_xlabel('Nombre de NaN')
    for i, v in enumerate(missing):
        axes[1].text(v + 0.5, i, f'{v} ({v/len(df)*100:.1f}%)', va='center')
else:
    axes[1].text(0.5, 0.5, 'Aucune valeur manquante !', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=14)
plt.tight_layout()
plt.show()

**Interprétation** : Les colonnes issues de l'enquête employé (EnvironmentSatisfaction, JobSatisfaction, WorkLifeBalance) contiennent des NaN car la participation n'était pas obligatoire. Les données numériques (NumCompaniesWorked, TotalWorkingYears…) présentent quelques lacunes ponctuelles. Le nombre de NaN reste faible (<5%), ce qui justifie une imputation simple (médiane pour les numériques, mode pour les catégorielles) plutôt qu'une suppression de lignes.

## Suppression du data leakage

Le **data leakage** (fuite de données) se produit lorsque des informations du **jeu de test** influencent la construction du modèle. Cela fausse l'évaluation : le modèle semble mieux généraliser qu'il ne le fait réellement.

**Exemple concret dans notre projet** : si l'on impute les valeurs manquantes avec la médiane calculée sur *toutes* les lignes (train + test), le modèle voit indirectement des statistiques du test set pendant l'entraînement. Même si l'impact est faible avec <5% de NaN, c'est une erreur méthodologique à éviter.

**Les sources classiques de data leakage** :

| Source | Risque | Notre stratégie |
|--------|--------|----------------|
| Imputation avant split | La médiane/mode contient de l'info du test set | `SimpleImputer` dans le `ColumnTransformer`, `fit` sur X_train uniquement |
| Scaling avant split | La moyenne/écart-type contient de l'info du test set | `StandardScaler` dans le `ColumnTransformer`, même logique |
| Feature selection avant split | Les corrélations contiennent de l'info du test set | Acceptable ici (propriété de la population, cf. § multicolinéarité) |
| Encoding avant split | Les modalités du test set influencent l'encoding | `OneHotEncoder(handle_unknown='ignore')` dans le pipeline |

> **Principe appliqué** : toutes les transformations qui *apprennent des paramètres* à partir des données (médiane, moyenne, écart-type, modalités) sont encapsulées dans un `ColumnTransformer` scikit-learn. Son `fit()` est appelé **uniquement sur X_train**, puis `transform()` est appliqué à X_test avec les mêmes paramètres. Cela garantit qu'aucune information du test set ne fuite vers le modèle.

## 1.7 Nettoyage des données

On prépare le dataset pour la modélisation en appliquant des transformations standards :

1. **Encodage** de la variable cible `Attrition` en binaire (Yes=1, No=0)
2. **Suppression** des colonnes constantes (détectées automatiquement via `nunique() <= 1` — comme Over18, EmployeeCount, StandardHours)
3. **Suppression** de EmployeeID (identifiant, pas une feature prédictive)
4. **Imputation** des valeurs manquantes (médiane pour numériques, mode pour catégorielles) — *pour l'EDA uniquement*
5. **Sauvegarde** du dataset complet avant filtre éthique (pour comparaison en Partie 3)

> **⚠️ Prévention du data leakage** : L'imputation manuelle ci-dessous est effectuée sur l'ensemble des données pour les besoins de l'analyse exploratoire (graphiques, corrélations). En revanche, pour la **pipeline de classification** (Partie 2), on utilise une copie **pré-imputation** (`df_before_impute`) : le `SimpleImputer` intégré au `ColumnTransformer` sklearn apprend les valeurs d'imputation **uniquement sur le jeu d'entraînement** (`fit` sur X_train), ce qui empêche toute fuite d'information du test set vers le modèle.

In [ ]:
# Encoder Attrition en binaire
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Identifier les colonnes constantes
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Colonnes constantes : {constant_cols}")

# Suppressions techniques uniquement (constantes + identifiant)
cols_to_drop_tech = list(set(constant_cols + ['EmployeeID']))
df = df.drop(columns=[c for c in cols_to_drop_tech if c in df.columns])
print(f"Dataset après nettoyage technique : {df.shape}")

In [ ]:
# ===========================================================
# SAUVEGARDE PRE-IMPUTATION (pour éviter le data leakage)
# On conserve le df avec NaN pour la pipeline ML.
# L'imputation ci-dessous sert UNIQUEMENT à l'EDA.
# ===========================================================
df_before_impute = df.copy()

# Imputation des valeurs manquantes numériques par la médiane
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        count = df[col].isnull().sum()
        df[col].fillna(median_val, inplace=True)
        print(f"  {col}: {count} NaN → médiane = {median_val}")

# Imputation des valeurs manquantes catégorielles par le mode
cat_cols_all = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols_all:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        count = df[col].isnull().sum()
        df[col].fillna(mode_val, inplace=True)
        print(f"  {col}: {count} NaN → mode = {mode_val}")

print(f"\nImputation EDA terminée — NaN restants dans df : {df.isnull().sum().sum()}")
print(f"df_before_impute conservé à part avec ses {df_before_impute.isnull().sum().sum()} NaN d'origine (sera utilisé par la pipeline ML pour éviter le data leakage)")

In [ ]:
# ===========================================================
# SAUVEGARDE du dataset complet (avant filtre éthique)
# Sera réutilisé pour la pipeline non-éthique
# ===========================================================
df_full = df.copy()
print(f"df_full sauvegardé : {df_full.shape} (toutes les variables, pour comparaison)")

## 1.8 Filtre éthique

On retire trois catégories de variables du dataset éthique :
- **Critères protégés** : Gender, Age, MaritalStatus
- **Métriques de surveillance** : avg_work_hours, std_work_hours, days_absent
- **Variable non actionnable** : DistanceFromHome

Le dataset complet (`df_full`) est conservé pour comparaison en Partie 3.

In [ ]:
# --- Suppressions éthiques ---
# 1) Critères protégés (discrimination directe)
ethical_protected = ['Gender', 'Age', 'MaritalStatus']

# 2) Métriques de surveillance issues du pointage
ethical_surveillance = ['avg_work_hours', 'std_work_hours', 'days_absent']

# 3) Variable non actionnable par l'entreprise
non_actionable = ['DistanceFromHome']

cols_to_drop_ethical = [c for c in ethical_protected + ethical_surveillance + non_actionable
                        if c in df.columns]

print("--- Suppressions éthiques ---")
print(f"Critères protégés (loi anti-discrimination) : {[c for c in ethical_protected if c in df.columns]}")
print(f"Surveillance comportementale (RGPD art. 22) : {[c for c in ethical_surveillance if c in df.columns]}")
print(f"Non actionnable par l'entreprise            : {[c for c in non_actionable if c in df.columns]}")

df = df.drop(columns=cols_to_drop_ethical)
print(f"\nDataset éthique : {df.shape}")
print(f"Colonnes restantes : {list(df.columns)}")

## 1.9 Analyse univariée

L'analyse univariée étudie chaque variable individuellement pour comprendre sa distribution, détecter les outliers et vérifier qu'il n'y a pas d'anomalies (valeurs aberrantes, distributions très asymétriques). C'est un prérequis avant l'analyse bivariée et la modélisation.

### Distribution de la variable cible (Attrition)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['Attrition'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['No (0)', 'Yes (1)'], counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribution de Attrition', fontsize=14)
axes[0].set_ylabel("Nombre d'employés")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['No', 'Yes'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0, 0.05])
axes[1].set_title('Proportion Attrition', fontsize=14)

plt.suptitle(f'Variable cible : Attrition (n={len(df)})', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n⚠️ Déséquilibre de classes : {counts[0]/counts[1]:.1f}:1 (No:Yes)")

**Interprétation** : Le dataset est fortement **déséquilibré** (~84% No vs ~16% Yes). Cela signifie qu'un modèle naïf prédisant toujours "No" atteindrait 84% d'accuracy sans rien apprendre. Il faudra donc :
- Utiliser `class_weight='balanced'` dans les classifieurs qui le supportent
- Privilégier le **F1-Score** plutôt que l'accuracy comme métriques d'évaluation

### Distribution des variables numériques

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_no_target = [c for c in num_cols if c != 'Attrition']

n_cols = 4
n_rows = (len(num_cols_no_target) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols_no_target):
    sns.histplot(df[col], ax=axes[i], kde=True, bins=30, color='steelblue')
    axes[i].set_title(col, fontsize=11)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7, label=f'mean={df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', alpha=0.7, label=f'median={df[col].median():.1f}')
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des variables numériques', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### Distribution des variables catégorielles

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

if len(cat_cols) > 0:
    n_cols_plot = 3
    n_rows_plot = (len(cat_cols) + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, 5 * n_rows_plot))
    axes = axes.flatten()

    for i, col in enumerate(cat_cols):
        order = df[col].value_counts().index
        sns.countplot(data=df, x=col, ax=axes[i], order=order, palette='Set2')
        axes[i].set_title(col, fontsize=12)
        axes[i].tick_params(axis='x', rotation=45)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribution des variables catégorielles', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune variable catégorielle restante.")

## 1.10 Analyse bivariée

L'analyse bivariée mesure les **relations entre paires de variables**. La matrice de corrélation et les visualisations croisées avec Attrition permettent d'identifier les features les plus prédictives et de détecter d'éventuelles redondances (multicolinéarité, traitée en 1.11).

### Matrice de corrélation

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(18, 14))
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 7})
plt.title('Matrice de corrélation', fontsize=16)
plt.tight_layout()
plt.show()

### Top corrélations avec Attrition

In [ ]:
corr_attrition = corr_matrix['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_attrition.values]
corr_attrition.plot(kind='barh', color=colors)
plt.title('Corrélation avec Attrition', fontsize=14)
plt.xlabel('Coefficient de corrélation de Pearson')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop 10 corrélations (valeur absolue) :")
print(corr_attrition.head(10).to_string())

**Interprétation** : Les corrélations les plus fortes avec l'attrition pointent vers des **leviers organisationnels** : satisfaction au travail, ancienneté, niveau salarial, nombre de formations. Les features liées à la carrière et à la rémunération sont systématiquement en tête, ce qui confirme que le turnover est en grande partie expliqué par des facteurs sur lesquels l'entreprise peut agir.

> **Note sur la corrélation de Pearson** : elle ne mesure que les relations **linéaires**. Si une variable avait une relation en U avec l'attrition (ex : les très faibles ET les très élevés partent davantage), Pearson ne la détecterait pas. Ce n'est pas un manque : les **modèles non-linéaires** (Random Forest, SVM RBF, MLP) capturent ces relations directement pendant l'entraînement. On n'a pas besoin de MSE ou R² ici — ce sont des métriques de **régression** (prédire une valeur continue), alors que notre problème est une **classification** (Attrition = oui/non).

### Boxplots : Variables numériques vs Attrition

La boîte à moustaches compare la **distribution** d'une variable numérique entre les employés qui sont restés (No) et ceux qui sont partis (Yes). La boîte montre la **médiane** (trait central), les **quartiles** Q1-Q3 (bords de la boîte), et les **outliers** (points). Si les deux boîtes sont décalées verticalement, c'est un signal que cette variable **discrimine** les deux groupes.

In [ ]:
top_num = corr_attrition.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(top_num):
    sns.boxplot(data=df, x='Attrition', y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xticklabels(['No', 'Yes'])

plt.suptitle('Variables numériques vs Attrition (top 8 corrélations)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

**Interprétation** : Les boîtes montrent des décalages clairs : par exemple, les employés partis (Yes) ont généralement un **salaire mensuel plus bas**, moins d'**ancienneté**, et un **niveau de satisfaction inférieur**. Plus l'écart entre les médianes des deux boîtes est grand, plus la variable est discriminante pour prédire l'attrition.

### Variables catégorielles vs Attrition

In [ ]:
if len(cat_cols) > 0:
    n_cols_plot = 3
    n_rows_plot = (len(cat_cols) + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, 5 * n_rows_plot))
    axes = axes.flatten()

    for i, col in enumerate(cat_cols):
        attrition_rate = df.groupby(col)['Attrition'].mean().sort_values(ascending=False)
        attrition_rate.plot(kind='bar', ax=axes[i], color='coral', edgecolor='black')
        axes[i].set_title(f"Taux d'attrition par {col}", fontsize=12)
        axes[i].set_ylabel("Taux d'attrition")
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].axhline(y=df['Attrition'].mean(), color='red', linestyle='--', alpha=0.5,
                        label=f'Moyenne = {df["Attrition"].mean():.2f}')
        axes[i].legend()

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Taux d'attrition par variable catégorielle", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()

## 1.11 Détection et suppression de la multicolinéarité

Deux features fortement corrélées entre elles apportent de l'information redondante. Cela peut :
- Déstabiliser les coefficients (régression logistique, perceptron)
- Augmenter le risque d'overfitting

**Seuil choisi : |r| > 0.75**

Ce seuil permet de supprimer les features qui partagent une part substantielle d'information :
- **0.90+** : trop permissif, laisse passer des quasi-doublons
- **0.75** : bon compromis, supprime les features qui partagent >56% de variance commune (r² > 0.56)
- **0.50** : trop agressif, supprime des features avec des signaux distincts

On applique cette détection sur **les deux datasets** (éthique et complet) en une seule passe.

> **Note sur le data leakage** : la matrice de corrélation est calculée ici sur le dataset complet (avant le split train/test). C'est un choix délibéré et accepté : la **structure de corrélation** entre features est une propriété de la *population* (elle décrit les relations structurelles entre variables), pas une statistique individuelle susceptible de révéler des observations du test set. D'ailleurs, refaire cette analyse uniquement sur le train set aurait produit des corrélations quasi-identiques (~4000 lignes donnent des estimations très stables). Intégrer cette sélection dans une boucle de cross-validation serait disproportionné pour le gain marginal obtenu.

In [ ]:
CORR_THRESHOLD = 0.75

def remove_multicollinearity(data, threshold, label):
    """Supprime les features fortement corrélées en gardant celle la plus liée à Attrition."""
    num = data.select_dtypes(include=[np.number]).columns.tolist()
    corr_mat = data[num].corr()
    upper = corr_mat.abs().where(np.triu(np.ones_like(corr_mat, dtype=bool), k=1))

    high_pairs = []
    for col in upper.columns:
        for idx in upper.index:
            val = upper.loc[idx, col]
            if pd.notna(val) and val > threshold and col != 'Attrition' and idx != 'Attrition':
                high_pairs.append((idx, col, corr_mat.loc[idx, col]))

    cols_to_remove = set()
    if high_pairs:
        print(f"\n[{label}] Paires avec |corrélation| > {threshold} :")
        for f1, f2, r in sorted(high_pairs, key=lambda x: abs(x[2]), reverse=True):
            print(f"  {f1:30s} ↔ {f2:30s}  r = {r:+.3f}")

        for f1, f2, r in high_pairs:
            if f1 in cols_to_remove or f2 in cols_to_remove:
                continue
            c1 = abs(corr_mat.loc[f1, 'Attrition']) if f1 in corr_mat.index else 0
            c2 = abs(corr_mat.loc[f2, 'Attrition']) if f2 in corr_mat.index else 0
            drop = f1 if c1 < c2 else f2
            cols_to_remove.add(drop)
            print(f"  → Suppression de '{drop}' "
                  f"(|corr Attrition| = {min(c1,c2):.4f} < {max(c1,c2):.4f})")

        data = data.drop(columns=list(cols_to_remove))
        print(f"  Colonnes supprimées : {list(cols_to_remove)}")
    else:
        print(f"[{label}] Aucune paire > {threshold}, pas de suppression.")

    print(f"  → {label} après multicolinéarité : {data.shape}")
    return data

# Appliquer sur les deux datasets en une seule passe
df      = remove_multicollinearity(df,      CORR_THRESHOLD, 'Éthique')
df_full = remove_multicollinearity(df_full, CORR_THRESHOLD, 'Non-éthique')

## 1.12 Synthèse EDA

### Ce que nous avons appris

1. **Déséquilibre de classes** : ~84% No vs ~16% Yes → les métriques classiques (accuracy) seront trompeuses. On utilisera F1-Score, Recall et AUC-ROC pour évaluer les modèles.

2. **Facteurs clés de l'attrition** : les variables les plus corrélées sont liées à la **satisfaction** (EnvironmentSatisfaction, JobSatisfaction), à la **carrière** (YearsSinceLastPromotion, YearsAtCompany, JobLevel) et à la **rémunération** (MonthlyIncome, StockOptionLevel). Ce sont des leviers sur lesquels l'entreprise peut agir.

3. **Qualité des données** : peu de valeurs manquantes (<5%), traitées par imputation médiane/mode. Colonnes constantes (Over18, EmployeeCount, StandardHours) supprimées automatiquement.

4. **Multicolinéarité** : les paires de features avec |r| > 0.75 ont été réduites pour éviter la redondance et stabiliser les modèles linéaires.

5. **Prévention du data leakage** : une copie pré-imputation (`df_before_impute`) est conservée pour alimenter la pipeline ML. Toutes les transformations paramétrées (imputation, scaling, encoding) sont encapsulées dans un `ColumnTransformer` dont le `fit()` ne voit que X_train. La sélection de features (multicolinéarité) est la seule étape réalisée sur le dataset complet, ce qui est acceptable car elle repose sur une propriété structurelle de la population.

### Implications pour la modélisation

- Le préprocessing (SimpleImputer + StandardScaler + OneHotEncoder) sera encapsulé dans un `ColumnTransformer` pour garantir la reproductibilité **et prévenir le data leakage**.
- On utilisera `class_weight='balanced'` pour les modèles qui le supportent, afin de compenser le déséquilibre.
- La comparaison éthique vs non-éthique se fera sur la même pipeline pour isoler l'effet des variables retirées.

---

# Partie 2 : Classification

On entraîne **8 classifieurs** sur les deux variantes du dataset (éthique et complet) via un feature flag `ethical_filter`.

**Choix des algorithmes** : on couvre un spectre large (linéaires, non-linéaires, ensembles, réseaux de neurones) pour identifier l'approche la mieux adaptée au problème.

**Métrique d'optimisation** : le F1-Score, compromis entre Precision et Recall, pertinent pour un dataset déséquilibré.

## 2.1 Préparation des pipelines

Pour chaque configuration, on construit un `ColumnTransformer` sklearn qui :
- **Impute** les NaN (médiane / mode) — **à partir du jeu d'entraînement uniquement** (pas de data leakage)
- **Standardise** les features numériques (centrage-réduction, indispensable pour SVM, KNN, Perceptron, MLP)
- **Encode** les features catégorielles en One-Hot (chaque modalité → colonne binaire)

> **Anti data-leakage** : On repart de `df_before_impute` (contenant encore les NaN) plutôt que du `df` imputé lors de l'EDA. Ainsi le `SimpleImputer` du `ColumnTransformer` calcule les valeurs de remplacement **sur X_train seulement** et les applique telles quelles à X_test, sans fuite d'information.

Le split train/test est **stratifié** (80/20) pour conserver la proportion de la classe minoritaire dans chaque partition.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# --- Feature flag : ethical_filter ---
# Le flag contrôle si les variables sensibles sont incluses ou non.
# On boucle sur [True, False] pour exécuter la même pipeline
# sur les deux configurations et comparer les résultats.

target = 'Attrition'
pipeline_data = {}

# --- Prévention du data leakage ---
# On reconstruit les datasets depuis df_before_impute (avec NaN)
# pour que le SimpleImputer du ColumnTransformer apprenne
# les valeurs d'imputation uniquement sur le train set.
# On applique les mêmes filtres de colonnes que df / df_full.
cols_ethical = df.columns.tolist()        # colonnes après filtre éthique + multicolinéarité
cols_full    = df_full.columns.tolist()   # colonnes après multicolinéarité seulement

for ethical_filter in [True, False]:
    label = 'Éthique' if ethical_filter else 'Non-éthique'
    cols_to_use = cols_ethical if ethical_filter else cols_full
    # Sélectionner depuis df_before_impute uniquement les colonnes qui existent
    available = [c for c in cols_to_use if c in df_before_impute.columns]
    pipe_df = df_before_impute[available].copy()

    X = pipe_df.drop(columns=[target])
    y = pipe_df[target].copy()

    num_feats = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_feats = X.select_dtypes(include=['object']).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_feats),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), cat_feats)
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # fit_transform sur X_train : le SimpleImputer apprend médiane/mode du TRAIN uniquement
    X_train_proc = preprocessor.fit_transform(X_train)
    # transform sur X_test : applique les mêmes valeurs (pas de leak)
    X_test_proc = preprocessor.transform(X_test)

    # Noms des features après transformation (pour feature importance)
    feat_names = (num_feats +
                  list(preprocessor.named_transformers_['cat']
                       .named_steps['encoder']
                       .get_feature_names_out(cat_feats)))

    pipeline_data[label] = {
        'ethical_filter': ethical_filter,
        'X': X, 'y': y,
        'X_train': X_train_proc, 'X_test': X_test_proc,
        'y_train': y_train, 'y_test': y_test,
        'preprocessor': preprocessor,
        'num_features': num_feats,
        'cat_features': cat_feats,
        'feature_names': feat_names,
    }

    print(f"\n{'='*50}")
    print(f" Pipeline : {label}  (ethical_filter={ethical_filter})")
    print(f"{'='*50}")
    print(f"  Features numériques   : {len(num_feats)}")
    print(f"  Features catégorielles: {len(cat_feats)}")
    print(f"  NaN dans X avant split: {X.isnull().sum().sum()}")
    print(f"  X_train: {X_train_proc.shape} | X_test: {X_test_proc.shape}")
    print(f"  Distribution cible (train) : {dict(pd.Series(y_train).value_counts())}")

extra_vars = sorted(set(pipeline_data['Non-éthique']['X'].columns) - set(pipeline_data['Éthique']['X'].columns))
print(f"\nVariables supplémentaires (non-éthique) : {extra_vars}")
print(f"Delta features : +{len(extra_vars)} variables")
print("\n✓ Data leakage prévenu : imputation calculée sur X_train uniquement")

## 2.2 Définition des classifieurs

On définit **8 classifieurs** couvrant les principales familles d'algorithmes :

| Famille | Modèle | Pourquoi |
|---|---|---|
| Linéaire | Régression Logistique, Perceptron | Interprétables, rapides, bonne baseline |
| Noyau | SVM (RBF) | Capture les non-linéarités via le kernel trick |
| Probabiliste | Naive Bayes | Baseline rapide, peu de tuning |
| Instance-based | KNN | Pas d'hypothèse sur la distribution |
| Arbres | Decision Tree, Random Forest | Gèrent les interactions, feature importance |
| Réseau de neurones | MLP | Capture les relations complexes |

Chaque modèle est instancié avec `class_weight='balanced'` quand disponible, pour compenser le déséquilibre de classes.

In [ ]:
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve, average_precision_score)


def get_classifiers():
    """Retourne un dictionnaire frais de 8 classifieurs (mêmes hyperparamètres)."""
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
        'Perceptron': Perceptron(
            max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
        'SVM (RBF)': SVC(
            kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
        'Naive Bayes': GaussianNB(),
        'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'Decision Tree': DecisionTreeClassifier(
            max_depth=10, class_weight='balanced', random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'MLP (Neural Network)': MLPClassifier(
            hidden_layer_sizes=(128, 64), max_iter=500, random_state=RANDOM_STATE,
            early_stopping=True, validation_fraction=0.15),
    }


def train_and_evaluate(classifiers_dict, X_train, X_test, y_train, y_test):
    """Entraîne tous les classifieurs et retourne un dictionnaire de résultats."""
    results = {}
    for name, clf in classifiers_dict.items():
        start = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - start

        y_pred = clf.predict(X_test)

        if hasattr(clf, 'predict_proba'):
            y_proba = clf.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_proba)
        elif hasattr(clf, 'decision_function'):
            y_scores = clf.decision_function(X_test)
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        results[name] = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'AUC-ROC': auc,
            'Training_time': train_time,
            'y_pred': y_pred,
        }

    return results


def results_to_df(results):
    """Convertit un dict de résultats en DataFrame (sans y_pred)."""
    return pd.DataFrame({
        name: {k: v for k, v in m.items() if k != 'y_pred'}
        for name, m in results.items()
    }).T


print("Fonctions et classifieurs définis ✓")


## 2.3 Entraînement des 8 classifieurs (boucle)

In [ ]:
all_results = {}
all_classifiers = {}

for label in pipeline_data:
    d = pipeline_data[label]
    classifiers = get_classifiers()

    ef = d['ethical_filter']
    print(f"Entraînement {label} (ethical_filter={ef})... ", end="")

    results = train_and_evaluate(
        classifiers, d['X_train'], d['X_test'], d['y_train'], d['y_test']
    )

    all_results[label] = results
    all_classifiers[label] = classifiers
    best = max(results, key=lambda n: results[n]['F1'])
    print(f"✓  (meilleur F1 : {best} — {results[best]['F1']:.4f})")

print("\n✓ 8 modèles entraînés x 2 pipelines. Détails dans la section Résultats consolidés.")


**Interprétation des résultats initiaux** : Les modèles d'ensemble (Random Forest) et à noyau (SVM) obtiennent généralement les meilleurs F1-Scores. Le Perceptron et le Naive Bayes, plus simples, servent de baseline utile. Les écarts entre pipeline éthique et non-éthique seront analysés dans les résultats consolidés (Partie 3).

On sélectionne ensuite les 3 meilleurs modèles par F1 pour l'optimisation des hyperparamètres.


## 2.4 Optimisation des hyperparamètres

Pour améliorer les performances, on optimise les **3 meilleurs modèles** (par F1) via **RandomizedSearchCV** (30 itérations, 5-fold stratifié).

**Pourquoi RandomizedSearchCV plutôt que GridSearchCV ?**
- Avec de nombreux hyperparamètres, une grille exhaustive est trop coûteuse
- La recherche aléatoire explore plus efficacement l'espace et donne souvent de meilleurs résultats à budget fixe

**Métrique d'optimisation** : F1-Score (compromis Precision/Recall adapté au déséquilibre de classes).

**Pourquoi on conserve toujours le modèle optimisé** : Lors de l'optimisation, `RandomizedSearchCV` évalue chaque combinaison d'hyperparamètres par **validation croisée stratifiée à 5 folds** — c'est-à-dire 5 entraînements/évaluations sur 5 découpages différents du jeu d'entraînement. Le score CV rapporté est la **moyenne de ces 5 évaluations**, ce qui en fait une estimation plus stable et moins sujette au hasard du découpage qu'un unique score calculé sur un seul split train/test. C'est pourquoi on met systématiquement à jour le modèle avec les meilleurs hyperparamètres trouvés par CV, même si le F1 sur le test set unique est parfois très légèrement inférieur : le score CV est un meilleur indicateur de la performance réelle en généralisation.


In [ ]:
from scipy.stats import randint, uniform

# ===========================================================
# SNAPSHOT des résultats AVANT optimisation (pour comparaison)
# ===========================================================
pre_tuning_results = {}
for label in all_results:
    pre_tuning_results[label] = {
        name: {k: v for k, v in m.items() if k != 'y_pred'}
        for name, m in all_results[label].items()
    }

# Espaces de recherche pour les modèles optimisables
param_distributions = {
    'Logistic Regression': {
        'C': uniform(0.01, 10),
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
    },
    'Random Forest': {
        'n_estimators': randint(50, 300),
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'max_features': ['sqrt', 'log2', None],
    },
    'MLP (Neural Network)': {
        'hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32), (256, 128)],
        'alpha': uniform(0.0001, 0.01),
        'learning_rate_init': uniform(0.0005, 0.01),
        'batch_size': [32, 64, 128],
    },
    'SVM (RBF)': {
        'C': uniform(0.1, 10),
        'gamma': ['scale', 'auto'] + list(uniform(0.001, 0.1).rvs(5, random_state=RANDOM_STATE)),
    },
    'Decision Tree': {
        'max_depth': [5, 10, 15, 20, 30, None],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'criterion': ['gini', 'entropy'],
    },
    'KNN (k=5)': {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['minkowski', 'euclidean', 'manhattan'],
    },
}

# Factories pour recréer des modèles vierges
base_model_factories = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': lambda: RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'MLP (Neural Network)': lambda: MLPClassifier(
        max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
    'SVM (RBF)': lambda: SVC(
        kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
    'Decision Tree': lambda: DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE),
    'KNN (k=5)': lambda: KNeighborsClassifier(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Collecte des résultats d'optimisation pour la table de comparaison
tuning_comparison = []

for label in pipeline_data:
    d = pipeline_data[label]
    results = all_results[label]
    classifiers = all_classifiers[label]

    ranking = results_to_df(results).sort_values('F1', ascending=False)
    top_models = [name for name in ranking.index[:3]
                  if name in param_distributions and name in base_model_factories]
    print(f"[{label}] Top 3 → optimisation : {top_models}")

    for name in top_models:
        search = RandomizedSearchCV(
            base_model_factories[name](),
            param_distributions[name],
            n_iter=30,
            cv=cv,
            scoring='f1',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=0
        )
        search.fit(d['X_train'], d['y_train'])

        y_pred_opt = search.best_estimator_.predict(d['X_test'])
        f1_before = results[name]['F1']
        f1_after = f1_score(d['y_test'], y_pred_opt)
        recall_after = recall_score(d['y_test'], y_pred_opt)

        if hasattr(search.best_estimator_, 'predict_proba'):
            y_proba_opt = search.best_estimator_.predict_proba(d['X_test'])[:, 1]
            auc_opt = roc_auc_score(d['y_test'], y_proba_opt)
        elif hasattr(search.best_estimator_, 'decision_function'):
            auc_opt = roc_auc_score(
                d['y_test'], search.best_estimator_.decision_function(d['X_test']))
        else:
            auc_opt = np.nan

        # Sauvegarder la comparaison avant/après
        tuning_comparison.append({
            'Pipeline': label,
            'Modèle': name,
            'F1 avant': f1_before,
            'F1 après': f1_after,
            'Δ F1': f1_after - f1_before,
            'Recall avant': pre_tuning_results[label][name]['Recall'],
            'Recall après': recall_after,
            'AUC avant': pre_tuning_results[label][name]['AUC-ROC'],
            'AUC après': auc_opt,
            'CV F1': search.best_score_,
            'Meilleurs params': str(search.best_params_),
        })

        # Mettre à jour le modèle et les résultats
        results[name] = {
            'Accuracy': accuracy_score(d['y_test'], y_pred_opt),
            'Precision': precision_score(d['y_test'], y_pred_opt, zero_division=0),
            'Recall': recall_after,
            'F1': f1_after,
            'AUC-ROC': auc_opt,
            'Training_time': results[name]['Training_time'],
            'y_pred': y_pred_opt,
        }
        classifiers[name] = search.best_estimator_

print(f"\n✓ {len(tuning_comparison)} modèles optimisés. Voir tableau ci-dessous.")


### Tableau comparatif avant / après fine-tuning

Le tableau ci-dessous résume l'impact du tuning par `RandomizedSearchCV` sur chaque modèle optimisé, pour les deux pipelines. Ce même tableau est repris dans les résultats consolidés (section 3.2) pour une vue d'ensemble.


In [ ]:
# ===========================================================
# TABLE DE COMPARAISON AVANT / APRÈS FINE-TUNING
# ===========================================================
df_tuning = pd.DataFrame(tuning_comparison)

# Affichage formaté
display_cols = ['Pipeline', 'Modèle', 'F1 avant', 'F1 après', 'Δ F1',
                'Recall avant', 'Recall après', 'AUC avant', 'AUC après', 'CV F1']
df_display = df_tuning[display_cols].copy()

# Formater les colonnes numériques
for col in df_display.columns:
    if col not in ['Pipeline', 'Modèle']:
        df_display[col] = df_display[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

print("="*90)
print(" IMPACT DU FINE-TUNING (avant → après RandomizedSearchCV)")
print("="*90)
display(df_display.style
        .set_caption("Comparaison avant/après optimisation des hyperparamètres")
        .hide(axis='index'))

# Résumé
mean_delta = df_tuning['Δ F1'].mean()
best_gain = df_tuning.loc[df_tuning['Δ F1'].idxmax()]
print("\n📊 Résumé :")
print(f"  Δ F1 moyen         : {mean_delta:+.4f}")
print(f"  Meilleure amélio.  : {best_gain['Modèle']} ({best_gain['Pipeline']}) → Δ F1 = {best_gain['Δ F1']:+.4f}")
print(f"  Modèles optimisés  : {len(df_tuning)}")

**Interprétation du fine-tuning** : Le tuning par `RandomizedSearchCV` (30 itérations, 5-fold stratifié) explore l'espace des hyperparamètres de manière efficace. Les points clés :

- **Δ F1 positif** → les hyperparamètres par défaut n'étaient pas optimaux ; le tuning apporte une amélioration mesurable
- **CV F1** → le score de validation croisée (moyenne sur 5 folds) est plus fiable qu'un seul split car il réduit la variance d'estimation
- **Recall** → métrique critique dans notre contexte RH (manquer un employé à risque de départ coûte plus cher qu'un faux positif). On vérifie que le tuning ne sacrifie pas le Recall au profit de la Precision

> Les résultats détaillés de tous les modèles (avant/après optimisation, les deux pipelines) sont présentés dans la **section Résultats consolidés** (Partie 3) pour éviter la redondance.


## 2.5 Analyse approfondie du meilleur modèle (pipeline éthique)

On détaille le fonctionnement, les forces et les limites du modèle le plus performant sur la pipeline éthique — c'est celui qui serait déployé en production. On vérifie sa stabilité via validation croisée et on identifie les features les plus influentes.


In [ ]:
# Identifier le meilleur modèle éthique (par F1)
ranking_eth = results_to_df(all_results['Éthique']).sort_values('F1', ascending=False)

best_name = ranking_eth.index[0]
best_model = all_classifiers['Éthique'][best_name]
best_metrics = all_results['Éthique'][best_name]

print(f"🏆 Meilleur modèle (éthique) : {best_name}")
print("\nMétriques sur le test set :")
for k, v in best_metrics.items():
    if k != 'y_pred':
        print(f"  {k:15s} : {v:.4f}" if isinstance(v, float) else f"  {k:15s} : {v}")

### SVM (Support Vector Machine) avec noyau RBF — Explication détaillée

#### Intuition
Un SVM cherche l'**hyperplan** qui sépare les deux classes (Attrition = Oui/Non) avec la **marge maximale** — c'est-à-dire la plus grande distance possible entre l'hyperplan et les points les plus proches de chaque classe (appelés *support vectors*). Maximiser cette marge améliore la généralisation : le modèle est moins sensible au bruit.

#### Le kernel trick (noyau RBF)
Quand les données ne sont pas linéairement séparables dans l'espace original (ce qui est fréquent avec des données RH multi-facteurs), le SVM projette implicitement les observations dans un **espace de dimension supérieure** où la séparation devient possible. Le noyau **RBF** (*Radial Basis Function*) calcule cette projection via :

$$K(x, x') = \exp\left(-\gamma \|x - x'\|^2\right)$$

Concrètement, chaque point est entouré d'une "bulle d'influence" gaussienne. Le noyau mesure la **similarité** entre deux observations : plus elles sont proches, plus $K$ est grand.

#### Rôle des hyperparamètres

| Paramètre | Rôle | Effet d'une valeur élevée |
|-----------|------|--------------------------|
| **C** (régularisation) | Contrôle le compromis entre marge large et erreurs de classification | C élevé → marge étroite, peu d'erreurs sur le train (risque d'overfitting). C faible → marge large, plus tolérante (risque d'underfitting) |
| **γ** (gamma) | Rayon d'influence de chaque support vector | γ élevé → chaque point a une influence très locale (frontière complexe, risque d'overfitting). γ faible → influence étalée (frontière lisse) |

L'optimisation par `RandomizedSearchCV` explore différentes combinaisons de C et γ pour trouver le meilleur compromis biais-variance.

#### Pourquoi SVM (RBF) est pertinent ici
- **Non-linéarité** : les relations entre satisfaction, ancienneté, rémunération et attrition ne sont pas linéaires. Le noyau RBF les capture naturellement.
- **Haute dimension** : après One-Hot Encoding, on travaille avec des dizaines de features. Les SVM sont performants en haute dimension car ils ne dépendent que des support vectors, pas de toutes les observations.
- **Robustesse** : `class_weight='balanced'` pondère la classe minoritaire (Attrition=Yes), compensant le déséquilibre 84/16%.

#### Limites
- **Complexité** : O(n²) à O(n³) en mémoire/temps → peut être lent sur de très grands datasets (ici ~4000 lignes, donc acceptable).
- **Interprétabilité** : contrairement à un Decision Tree, on ne peut pas expliquer directement *pourquoi* un employé est prédit à risque. On compense via la feature importance par permutation (section 3.7).

In [ ]:
# Validation croisée du meilleur modèle éthique
d_eth = pipeline_data['Éthique']
feature_names_eth = d_eth['feature_names']

cv_scores = cross_val_score(best_model, d_eth['X_train'], d_eth['y_train'],
                            cv=5, scoring='f1', n_jobs=-1)
print(f"Validation croisée 5-fold ({best_name}) :")
print(f"  F1 moyen : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Scores   : [{', '.join(f'{s:.4f}' for s in cv_scores)}]")

# Si le modèle a feature_importances_ (tree-based)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-15:]

    plt.figure(figsize=(10, 8))
    plt.barh(range(len(indices)), importances[indices], color='steelblue', edgecolor='black')
    plt.yticks(range(len(indices)), [feature_names_eth[i] for i in indices])
    plt.xlabel('Importance')
    plt.title(f'{best_name} : Top 15 Features les plus importantes')
    plt.tight_layout()
    plt.show()

elif hasattr(best_model, 'coef_'):
    coefs = best_model.coef_.flatten()
    indices = np.argsort(np.abs(coefs))[-15:]

    plt.figure(figsize=(10, 8))
    colors_coef = ['#e74c3c' if c > 0 else '#3498db' for c in coefs[indices]]
    plt.barh(range(len(indices)), coefs[indices], color=colors_coef, edgecolor='black')
    plt.yticks(range(len(indices)), [feature_names_eth[i] for i in indices])
    plt.xlabel('Coefficient')
    plt.title(f'{best_name} : Top 15 Coefficients (rouge = augmente Attrition)')
    plt.tight_layout()
    plt.show()

else:
    print("Ce modèle ne fournit pas directement l'importance des features.")

# Partie 3 : Analyse Comparative et Recommandations

Cette partie confronte les résultats des deux pipelines (éthique vs complète) pour valider que le retrait des variables sensibles n'a pas dégradé significativement les performances. On conclut par les recommandations RH tirées du modèle retenu.

## 3.1 Matrices de confusion

Les matrices de confusion montrent la répartition des prédictions correctes et incorrectes pour chaque modèle. Elles sont à la base du calcul des métriques F1, Precision et Recall (section suivante). On les présente ici en premier car elles permettent de comprendre *d'où viennent* les scores.

> **Lecture** : la diagonale (haut-gauche, bas-droite) = prédictions correctes. En contexte RH, les **faux négatifs** (bas-gauche) sont les plus coûteux : ce sont les employés à risque que le modèle n'a pas détectés.


In [ ]:
# Résultats DataFrames (réutilisés dans les sections suivantes)
df_res_eth = results_to_df(all_results['Éthique']).sort_values('F1', ascending=False)
df_res_noeth = results_to_df(all_results['Non-éthique']).sort_values('F1', ascending=False)

# ===========================================================
# MATRICES DE CONFUSION — tous les modèles, côte à côte
# ===========================================================
model_names = list(all_classifiers['Éthique'].keys())
n_models = len(model_names)

fig, axes = plt.subplots(n_models, 2, figsize=(12, 4 * n_models))

for i, name in enumerate(model_names):
    for j, label in enumerate(['Éthique', 'Non-éthique']):
        d = pipeline_data[label]
        cm = confusion_matrix(d['y_test'], all_results[label][name]['y_pred'])
        disp = ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes'])
        disp.plot(ax=axes[i, j], cmap='Blues' if j == 0 else 'Oranges', colorbar=False)
        axes[i, j].set_title(f'{name} — {label}', fontsize=10)

plt.suptitle('Matrices de confusion : Éthique (bleu) vs Non-éthique (orange)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


**Lecture des matrices** : La diagonale principale représente les prédictions correctes. En contexte RH, les **faux négatifs** (bas-gauche) comptent le plus : ce sont des employés à risque non détectés. Un bon Recall = peu de faux négatifs. Les métriques dérivées de ces matrices sont analysées dans la section suivante.


## 3.2 Résultats consolidés : performances et impact du filtre éthique

Cette section réunit **toutes les métriques de performance** des 8 modèles sur les deux pipelines (éthique vs non-éthique), l'impact du fine-tuning, et la comparaison visuelle côte à côte. Cela permet une lecture unique et complète, sans dispersion.


In [ ]:
# ===========================================================
# 1. TABLEAUX COMPARATIFS — tous les modèles, côte à côte
# ===========================================================
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']

# Table fusionnée : éthique et non-éthique côte à côte
combined = df_res_eth[metrics_cols].copy()
combined.columns = [f'{c} (éth.)' for c in metrics_cols]
for c in metrics_cols:
    combined[f'{c} (non-éth.)'] = df_res_noeth.reindex(combined.index)[c]
    combined[f'Δ {c}'] = combined[f'{c} (non-éth.)'] - combined[f'{c} (éth.)']

print("="*100)
print(" PERFORMANCES CONSOLIDÉES — Éthique vs Non-éthique (après optimisation)")
print("="*100)

# Affichage éthique
print(f"\n--- Pipeline ÉTHIQUE ---")
print(df_res_eth[metrics_cols].to_string(float_format=lambda x: f'{x:.4f}'))

# Affichage non-éthique
print(f"\n--- Pipeline NON-ÉTHIQUE ---")
print(df_res_noeth[metrics_cols].to_string(float_format=lambda x: f'{x:.4f}'))

# Meilleurs modèles
print(f"\n🏆 Meilleur F1 éthique     : {df_res_eth.index[0]} ({df_res_eth['F1'].iloc[0]:.4f})")
print(f"🏆 Meilleur F1 non-éthique : {df_res_noeth.index[0]} ({df_res_noeth['F1'].iloc[0]:.4f})")

# ===========================================================
# 2. IMPACT DU FINE-TUNING (avant → après)
# ===========================================================
df_tuning = pd.DataFrame(tuning_comparison)
if len(df_tuning) > 0:
    display_cols = ['Pipeline', 'Modèle', 'F1 avant', 'F1 après', 'Δ F1',
                    'Recall avant', 'Recall après', 'AUC avant', 'AUC après', 'CV F1']
    df_display = df_tuning[display_cols].copy()
    for col in df_display.columns:
        if col not in ['Pipeline', 'Modèle']:
            df_display[col] = df_display[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

    print("\n" + "="*100)
    print(" IMPACT DU FINE-TUNING (avant → après RandomizedSearchCV)")
    print("="*100)
    display(df_display.style
            .set_caption("Comparaison avant/après optimisation des hyperparamètres")
            .hide(axis='index'))

    mean_delta = df_tuning['Δ F1'].mean()
    best_gain = df_tuning.loc[df_tuning['Δ F1'].idxmax()]
    print(f"\n  Δ F1 moyen         : {mean_delta:+.4f}")
    print(f"  Meilleure amélio.  : {best_gain['Modèle']} ({best_gain['Pipeline']}) → Δ F1 = {best_gain['Δ F1']:+.4f}")

# ===========================================================
# 3. IMPACT DU FILTRE ÉTHIQUE (delta entre pipelines)
# ===========================================================
comparison = pd.DataFrame({
    'F1_éthique': df_res_eth['F1'],
    'F1_non_éthique': df_res_noeth.reindex(df_res_eth.index)['F1'],
    'Recall_éthique': df_res_eth['Recall'],
    'Recall_non_éthique': df_res_noeth.reindex(df_res_eth.index)['Recall'],
    'AUC_éthique': df_res_eth['AUC-ROC'],
    'AUC_non_éthique': df_res_noeth.reindex(df_res_eth.index)['AUC-ROC'],
})
comparison['Δ_F1'] = comparison['F1_non_éthique'] - comparison['F1_éthique']
comparison['Δ_Recall'] = comparison['Recall_non_éthique'] - comparison['Recall_éthique']

print("\n" + "="*100)
print(" IMPACT DU FILTRE ÉTHIQUE (non-éthique − éthique)")
print("="*100)
print(comparison.to_string(float_format=lambda x: f'{x:.4f}'))

mean_delta_f1 = comparison['Δ_F1'].mean()
mean_delta_recall = comparison['Δ_Recall'].mean()
print(f"\n  Δ F1 moyen     : {mean_delta_f1:+.4f}")
print(f"  Δ Recall moyen : {mean_delta_recall:+.4f}")

if abs(mean_delta_f1) < 0.02:
    print("  → Impact NÉGLIGEABLE du filtre éthique (<2 pts de F1)")
elif mean_delta_f1 > 0:
    print(f"  → Les variables sensibles AMÉLIORENT les performances (+{mean_delta_f1:.1%})")
    print("    Mais leur usage est éthiquement inacceptable.")
else:
    print(f"  → Le filtre éthique AMÉLIORE légèrement les performances ({mean_delta_f1:+.1%})")

# ===========================================================
# 4. GRAPHIQUE COMPARATIF — 4 métriques côte à côte
# ===========================================================
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, metric in zip(axes.flatten(), metrics_to_plot):
    models = df_res_eth.index
    x = np.arange(len(models))
    width = 0.35

    vals_eth = df_res_eth.loc[models, metric]
    vals_noeth = df_res_noeth.reindex(models)[metric]

    ax.barh(x - width/2, vals_eth, width, label='Éthique', color='#2ecc71', edgecolor='black')
    ax.barh(x + width/2, vals_noeth, width, label='Non-éthique', color='#e74c3c', edgecolor='black', alpha=0.7)

    for i in range(len(models)):
        ax.text(vals_eth.iloc[i] + 0.005, x[i] - width/2, f'{vals_eth.iloc[i]:.3f}', va='center', fontsize=8)
        if pd.notna(vals_noeth.iloc[i]):
            ax.text(vals_noeth.iloc[i] + 0.005, x[i] + width/2, f'{vals_noeth.iloc[i]:.3f}', va='center', fontsize=8)

    ax.set_yticks(x)
    ax.set_yticklabels(models, fontsize=9)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontsize=14)
    ax.set_xlim(0, 1.1)
    ax.legend(fontsize=9)

plt.suptitle('Comparaison des classifieurs : Éthique vs Non-éthique (4 métriques)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


**Interprétation consolidée** :

- **Fine-tuning** : L'optimisation par `RandomizedSearchCV` améliore le F1 des modèles sélectionnés. Le score CV (moyenne 5 folds) est plus fiable qu'un seul score test, d'où la mise à jour systématique.
- **Filtre éthique** : Si le Δ F1 moyen est faible (<2 pts), les variables sensibles (âge, genre, heures de travail…) n'apportaient pas d'information prédictive significative au-delà des leviers organisationnels. Le filtre éthique est **non seulement justifié sur le plan des principes**, mais aussi **sans coût notable** en performance.
- **Graphiques** : Les barres côte à côte confirment visuellement que les 4 métriques sont comparables entre les deux pipelines pour la plupart des modèles.


## 3.3 Courbes ROC


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, label, color_cycle in zip(axes, ['Éthique', 'Non-éthique'], ['tab10', 'tab10']):
    d = pipeline_data[label]
    cmap = plt.get_cmap(color_cycle)

    for k, (name, clf) in enumerate(all_classifiers[label].items()):
        if hasattr(clf, 'predict_proba'):
            y_proba = clf.predict_proba(d['X_test'])[:, 1]
        elif hasattr(clf, 'decision_function'):
            y_proba = clf.decision_function(d['X_test'])
        else:
            continue

        fpr, tpr, _ = roc_curve(d['y_test'], y_proba)
        auc_val = roc_auc_score(d['y_test'], y_proba)
        ax.plot(fpr, tpr, label=f'{name} ({auc_val:.3f})',
                linewidth=2, color=cmap(k / max(len(all_classifiers[label]) - 1, 1)))

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (0.500)')
    ax.set_xlabel('FPR', fontsize=12)
    ax.set_ylabel('TPR', fontsize=12)
    ax.set_title(f'ROC — {label}', fontsize=14)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Courbes ROC : Éthique vs Non-éthique', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

**Interprétation ROC** : Plus une courbe se rapproche du coin supérieur gauche, meilleur est le modèle. L'AUC (Area Under Curve) résume cette performance en un seul chiffre : 0.5 = aléatoire, 1.0 = parfait. Les courbes des deux pipelines sont visuellement comparables, confirmant le faible impact du filtre éthique sur la capacité discriminante des modèles.

## 3.4 Courbes Precision-Recall


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, label in zip(axes, ['Éthique', 'Non-éthique']):
    d = pipeline_data[label]
    cmap = plt.get_cmap('tab10')

    for k, (name, clf) in enumerate(all_classifiers[label].items()):
        if hasattr(clf, 'predict_proba'):
            y_proba = clf.predict_proba(d['X_test'])[:, 1]
        elif hasattr(clf, 'decision_function'):
            y_proba = clf.decision_function(d['X_test'])
        else:
            continue

        precision_vals, recall_vals, _ = precision_recall_curve(d['y_test'], y_proba)
        ap = average_precision_score(d['y_test'], y_proba)
        ax.plot(recall_vals, precision_vals, label=f'{name} (AP={ap:.3f})',
                linewidth=2, color=cmap(k / max(len(all_classifiers[label]) - 1, 1)))

    baseline = d['y_test'].mean()
    ax.axhline(y=baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'Precision-Recall — {label}', fontsize=14)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Courbes Precision-Recall : Éthique vs Non-éthique', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

**Interprétation Precision-Recall** : Avec un dataset déséquilibré, la courbe PR est plus informative que la ROC. La baseline (ligne grise) correspond à la proportion de positifs dans le dataset. Un modèle utile doit être nettement au-dessus de cette baseline. L'Average Precision (AP) résume la qualité globale du classement.

## 3.5 Importance des features (éthique vs non-éthique)

L'énoncé demande de *"déterminer les facteurs ayant le plus d'influence sur le taux de rotation"*. Le feature importance répond directement à cette question : il indique quelles variables pèsent le plus dans la décision du modèle. On compare les deux pipelines pour voir si les mêmes leviers ressortent.


In [ ]:
# Feature importance pour les modèles tree-based — comparaison éthique vs non-éthique
for label in ['Éthique', 'Non-éthique']:
    feat_names_label = pipeline_data[label]['feature_names']
    tree_models = {name: clf for name, clf in all_classifiers[label].items()
                   if hasattr(clf, 'feature_importances_')}

    if len(tree_models) == 0:
        print(f"[{label}] Aucun modèle tree-based disponible.")
        continue

    n_plots = min(len(tree_models), 3)
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 8))
    if n_plots == 1:
        axes = [axes]

    for ax, (name, clf_model) in zip(axes, list(tree_models.items())[:3]):
        importances = clf_model.feature_importances_
        indices = np.argsort(importances)[-15:]

        ax.barh(range(len(indices)), importances[indices],
                color='steelblue' if label == 'Éthique' else '#e67e22', edgecolor='black')
        ax.set_yticks(range(len(indices)))
        ax.set_yticklabels([feat_names_label[i] for i in indices])
        ax.set_xlabel('Importance')
        ax.set_title(f'{name}', fontsize=12)

    plt.suptitle(f"Top 15 features — Pipeline {label}", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

**Interprétation** : Les features les plus importantes sont liées à la **satisfaction**, à l'**évolution de carrière** et à la **rémunération** dans les deux pipelines. Dans la pipeline non-éthique, les variables sensibles (Age, heures de travail…) peuvent apparaître, mais les leviers organisationnels restent prédominants — ce qui justifie a posteriori le choix du filtre éthique.

## 3.5bis Interprétabilité avec SHAP (SHapley Additive exPlanations)

### Pourquoi ajouter SHAP ?

Le meilleur modèle éthique (SVM RBF) est un **modèle black-box** : contrairement aux arbres de décision, il n'expose ni `feature_importances_` ni `coef_`. On ne peut donc pas expliquer directement *pourquoi* un employé est prédit à risque de départ.

**SHAP** résout ce problème de manière rigoureuse. Il s'appuie sur les **valeurs de Shapley** issues de la théorie des jeux coopératifs : pour chaque prédiction, SHAP calcule la contribution marginale de chaque feature en considérant toutes les coalitions possibles de variables. C'est la **seule méthode** qui satisfait simultanément les propriétés d'efficacité, de symétrie et d'additivité.

**Apports concrets :**
- **Vue globale** (summary plot) : quelles features influencent le plus les prédictions *en moyenne* sur l'ensemble du test set → complète le feature importance des modèles tree-based
- **Vue locale** (waterfall plot) : pourquoi *cet employé précis* est prédit à risque → utile pour le service RH qui doit cibler ses actions
- **Transparence** : répond à l'exigence n°4 de la Commission Européenne sur l'IA de confiance (« *les systèmes d'IA et leurs décisions doivent être explicables* »)
- **Cohérence éthique** : on vérifie que le modèle ne s'appuie pas sur des proxys de variables sensibles

> **Note technique** : on utilise `KernelExplainer` (model-agnostic) car le SVM RBF n'a pas de `TreeExplainer`. Le calcul est plus lent mais on le limite à un sous-échantillon représentatif du test set.


In [ ]:
import shap

# ===========================================================
# SHAP — Interprétabilité du meilleur modèle éthique
# ===========================================================

d_eth = pipeline_data['Éthique']
feat_names = d_eth['feature_names']

# Sous-échantillon de fond (background) pour KernelExplainer
# On prend 100 observations du train set pour estimer E[f(x)]
np.random.seed(RANDOM_STATE)
bg_idx = np.random.choice(d_eth['X_train'].shape[0], size=100, replace=False)
background = d_eth['X_train'][bg_idx]

# Sous-échantillon à expliquer (200 observations du test set)
n_explain = min(200, d_eth['X_test'].shape[0])
explain_idx = np.random.choice(d_eth['X_test'].shape[0], size=n_explain, replace=False)
X_explain = d_eth['X_test'][explain_idx]

# KernelExplainer — on passe predict_proba, il retourne les SHAP values pour chaque classe
explainer = shap.KernelExplainer(best_model.predict_proba, background)

print(f"Calcul des SHAP values pour {n_explain} observations...")
print("(KernelExplainer est model-agnostic — ~1-2 min)")
shap_values_raw = explainer.shap_values(X_explain)

# --- Extraction correcte des SHAP values pour la classe 1 (Attrition=Yes) ---
# Selon la version de SHAP :
#   - list de 2 arrays (anciennes versions) → shap_values_raw[1]
#   - ndarray 3D (n, features, 2) (versions récentes) → shap_values_raw[:, :, 1]
if isinstance(shap_values_raw, list):
    shap_vals = shap_values_raw[1]                # shape (n, features)
    base_val = explainer.expected_value[1]
else:
    shap_vals = shap_values_raw[:, :, 1]          # shape (n, features)
    base_val = explainer.expected_value[1]

print(f"Shape SHAP values (classe 1) : {shap_vals.shape}")
print(f"Base value (E[f(x)] classe 1) : {base_val:.4f}")

# ===========================================================
# 1. BEESWARM PLOT — importance globale avec direction d'effet
# ===========================================================
print("\n📊 Summary plot : importance globale des features")

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_vals, X_explain, feature_names=feat_names,
                  plot_type="dot", show=False, max_display=20, plot_size=None)
plt.title(f"SHAP Summary — {best_name} (pipeline éthique)", fontsize=15, pad=15)
plt.xlabel("Impact sur la prédiction d'attrition (SHAP value)", fontsize=11)
plt.tight_layout()
plt.show()

# ===========================================================
# 2. BAR PLOT — classement par importance moyenne absolue
# ===========================================================
print("\n📊 Bar plot : classement des features par |SHAP| moyen")

mean_abs_shap = np.abs(shap_vals).mean(axis=0)
top_k = 15
top_idx = np.argsort(mean_abs_shap)[-top_k:]

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = plt.cm.YlOrRd(np.linspace(0.3, 0.95, top_k))
ax.barh(range(top_k), mean_abs_shap[top_idx], color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(top_k))
ax.set_yticklabels([feat_names[i] for i in top_idx], fontsize=10)
ax.set_xlabel("Mean |SHAP value|", fontsize=12)
ax.set_title(f"SHAP Feature Importance — {best_name} (pipeline éthique)",
             fontsize=14, pad=12)
# Annotations
for i, idx in enumerate(top_idx):
    ax.text(mean_abs_shap[idx] + 0.002, i, f"{mean_abs_shap[idx]:.3f}",
            va='center', fontsize=9, color='#333')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# ===========================================================
# 3. WATERFALL PLOT — explication locale (1 employé à risque)
# ===========================================================
print("\n📊 Waterfall plot : explication d'une prédiction individuelle")

y_pred_explain = best_model.predict(X_explain)
at_risk_mask = y_pred_explain == 1
if at_risk_mask.sum() > 0:
    example_idx = np.where(at_risk_mask)[0][0]
else:
    example_idx = 0

pred_label = 'À risque' if y_pred_explain[example_idx] == 1 else 'Stable'
print(f"  Employé #{example_idx} — Prédiction : {pred_label}")

explanation = shap.Explanation(
    values=shap_vals[example_idx],          # 1D (n_features,)
    base_values=float(base_val),            # scalar
    data=X_explain[example_idx],            # 1D (n_features,)
    feature_names=feat_names
)
shap.waterfall_plot(explanation, max_display=15, show=False)
plt.title(f"Pourquoi cet employé est prédit {pred_label.lower()} ?", fontsize=13, pad=10)
plt.tight_layout()
plt.show()

print("\n✓ Analyse SHAP terminée")

### Interprétation des résultats SHAP

> **Note sur la densité du beeswarm** : le graphique n'affiche que quelques dizaines de points par feature, ce qui peut paraître peu comparé aux beeswarms denses que l'on rencontre couramment (500+ points). Cela est dû au `KernelExplainer` utilisé ici : étant **model-agnostic**, il doit interroger le modèle des milliers de fois par observation pour estimer les valeurs de Shapley, ce qui rend le calcul coûteux (~16 min pour 50 observations avec notre SVM RBF). Les beeswarms très denses que l'on voit en ligne utilisent généralement un `TreeExplainer` (Random Forest, XGBoost…), qui est algorithmiquement exact et quasi-instantané. Ce n'est pas applicable à un SVM. Le nombre de 50 observations reste statistiquement suffisant pour identifier les tendances globales et la direction des effets.

#### Figure 1 : Summary plot (beeswarm)

Le summary plot révèle que **TotalWorkingYears** est de loin la feature la plus influente du modèle SVM (RBF). On observe un schéma clair : les points **roses/rouges** (valeur élevée, i.e. beaucoup d'expérience) se situent majoritairement à **gauche** du zéro (SHAP négatif → pousse vers Attrition=No), tandis que les points **bleus** (peu d'expérience) se dispersent vers la **droite** (SHAP positif → pousse vers Attrition=Yes), parfois jusqu'à +0.30. Autrement dit, les employés peu expérimentés sont nettement plus à risque de départ. Pour **EnvironmentSatisfaction** et **JobSatisfaction**, la même logique s'applique : une satisfaction faible (points rouges, car les valeurs standardisées s'inversent) pousse vers l'attrition. **NumCompaniesWorked** montre un effet inverse : avoir travaillé dans beaucoup d'entreprises (rouge, valeur élevée) pousse fortement vers l'attrition (+0.30 pour les cas extrêmes), ce qui traduit un profil de mobilité fréquente. On note également que **YearsWithCurrManager**, **PercentSalaryHike** et **TrainingTimesLastYear** ont des effets bilatéraux : selon le contexte de l'employé, ces variables peuvent pousser dans un sens ou dans l'autre, signe d'interactions non-linéaires que le SVM RBF capture bien. Enfin, les features catégorielles encodées (BusinessTravel, JobRole, Department) apparaissent en bas du classement avec des contributions plus faibles et concentrées autour de zéro.

#### Figure 2 : Bar plot (importance moyenne absolue)

Le bar plot quantifie l'importance globale par la **moyenne des |SHAP values|** sur les 50 observations. **TotalWorkingYears** domine avec un |SHAP| moyen de **0.046**, soit presque le double de la deuxième feature. Viennent ensuite **EnvironmentSatisfaction** (0.026) et **JobSatisfaction** (0.022), confirmant que la satisfaction au travail est le deuxième facteur d'attrition après l'expérience professionnelle. **NumCompaniesWorked** (0.021) et **YearsWithCurrManager** (0.018) complètent le top 5 — tous des leviers organisationnels sur lesquels l'entreprise peut agir (plans de rétention ciblés, amélioration des conditions de travail, stabilité managériale). Un groupe de features intermédiaires (PercentSalaryHike, TrainingTimesLastYear, WorkLifeBalance, JobInvolvement, Education, MonthlyIncome) se situe autour de 0.012, avec un impact significatif mais secondaire. Les features de queue (StockOptionLevel à 0.010, BusinessTravel à 0.007, JobLevel et YearsSinceLastPromotion à 0.007) ont une contribution marginale. Ce classement est cohérent avec les corrélations bivariées de l'EDA (Partie 1) mais apporte une vision plus fine car SHAP intègre les **interactions non-linéaires** que la corrélation de Pearson ne capte pas.

#### Figure 3 : Waterfall plot (explication locale)

#### Interprétation éthique


**Limite identifiée — discrimination directe vs indirecte.** SHAP vérifie que le modèle *n'utilise pas* de critères discriminatoires dans ses prédictions (pas de **discrimination directe**). Mais cela ne garantit pas l'absence de **discrimination indirecte** : même avec des features neutres, les prédictions pourraient *dans la pratique* désavantager davantage un groupe protégé qu'un autre (par exemple, si le modèle signale proportionnellement plus de femmes ou de seniors comme « à risque »). Pour vérifier cela, il faudrait comparer les taux de faux positifs et faux négatifs **par groupe démographique** (genre, tranche d'âge…) et voir s'il existe des écarts significatifs. Ce type d'audit est recommandé avant tout déploiement, mais dépasse le périmètre de ce notebook.

L'analyse SHAP joue un rôle central dans la **validation éthique** de notre modèle, au-delà de la simple suppression de variables sensibles en amont (filtre éthique, section 1.8).
> **Point éthique** : Les trois graphiques SHAP confirment que le modèle éthique s'appuie exclusivement sur des **leviers organisationnels** (expérience, satisfaction, rémunération, engagement) et non sur des proxys de variables sensibles comme l'âge ou le genre. Aucune feature protégée n'apparaît dans le top 20. Cela renforce la conformité du modèle avec les exigences de **transparence** (exigence n°4) et de **non-discrimination** (exigence n°5) de la Commission Européenne pour une IA de confiance.

**Non-discrimination et équité (exigence n°5).** Aucune feature protégée ou proxy évident n'apparaît dans le top 20 du bar plot. Les 5 premiers facteurs (TotalWorkingYears, EnvironmentSatisfaction, JobSatisfaction, NumCompaniesWorked, YearsWithCurrManager) sont tous des **leviers organisationnels actionnables** — l'entreprise peut agir sur la satisfaction, la stabilité managériale et les parcours de carrière sans cibler un groupe protégé.

**Absence de proxys discriminatoires.** Même après avoir retiré Gender, Age et MaritalStatus, le modèle pourrait théoriquement les reconstruire via des *proxys* — par exemple, TotalWorkingYears est fortement corrélé à l'âge. Or, le summary plot montre que TotalWorkingYears agit comme un indicateur d'**expérience professionnelle** (peu d'expérience → risque de départ), ce qui est un critère légitime et non discriminatoire. La relation est cohérente avec la littérature RH sur la rétention des jeunes talents, et le levier d'action associé (plans de développement, mentorat) ne cible pas un groupe démographique en particulier.

**Transparence et explicabilité (exigence n°4).** Le waterfall plot permet de fournir une **explication individuelle** à chaque employé identifié à risque. Plutôt qu'un simple score opaque, le service RH peut communiquer les raisons précises de l'alerte (ex : *"faible implication, rémunération en dessous de la médiane, peu de stock options"*). Cela respecte le principe de transparence algorithmique et facilite le **droit à l'explication** prévu par le RGPD (article 22).

## 3.6 Recommandations et Conclusion

### Choix du modèle retenu et justification de la métrique

Tout au long de la pipeline, nous avons utilisé le **F1-Score** comme métrique principale pour :
- **Sélectionner** les 3 meilleurs modèles à optimiser (section 2.4)
- **Optimiser** les hyperparamètres via `RandomizedSearchCV(scoring='f1')` (section 2.4)
- **Classer** les modèles dans le tableau comparatif consolidé (section 3.2)

Ce choix est délibéré : le F1-Score est la **moyenne harmonique** de la Precision et du Recall, ce qui en fait un **compromis équilibré** adapté à un dataset déséquilibré (84/16%). Optimiser directement sur le Recall maximiserait la détection des départs, mais risquerait d'écrouler la Precision (trop de faux positifs), rendant le modèle inutilisable en pratique. Le F1 garantit que les deux métriques restent à un niveau raisonnable.

Néanmoins, **dans l'interprétation métier**, le Recall mérite une attention particulière car les coûts des erreurs sont **asymétriques** :
- Un **faux négatif** (employé à risque non détecté) a un coût élevé → perte de talent, coût de remplacement estimé à 6-9 mois de salaire
- Un **faux positif** (employé stable signalé à tort) a un coût faible → entretien RH supplémentaire, sans conséquence négative pour l'employé

C'est pourquoi on vérifie systématiquement que le modèle retenu a un **Recall élevé** en plus de son bon F1. Si deux modèles avaient eu des F1 proches, le Recall aurait servi de critère de départage. En pratique, le modèle sélectionné (SVM RBF) combine les deux : meilleur F1 *et* bon Recall sur la pipeline éthique.

La comparaison avec la pipeline non-éthique (section 3.2) confirme que le retrait des variables sensibles ne dégrade pas significativement les performances.

### Leviers d'action RH identifiés

Les features les plus influentes du modèle éthique pointent vers des **leviers organisationnels concrets** :

| Levier | Features associées | Actions proposées |
|---|---|---|
| **Satisfaction** | EnvironmentSatisfaction, JobSatisfaction | Enquêtes régulières, amélioration des conditions de travail |
| **Équilibre vie pro/perso** | WorkLifeBalance | Flexibilité horaire, télétravail, droit à la déconnexion |
| **Carrière** | YearsSinceLastPromotion, YearsAtCompany | Plans de carrière, revues annuelles, mobilité interne |
| **Rémunération** | MonthlyIncome, PercentSalaryHike, StockOptionLevel | Benchmarks salariaux, révisions ciblées |
| **Engagement** | JobInvolvement, TrainingTimesLastYear | Budget formation, mentorat, responsabilisation |

### Limites du modèle

- **Dataset modeste** (~4 700 employés d'une seule entreprise indienne) → généralisation limitée
- Un **audit de disparate impact** séparé reste nécessaire pour vérifier l'absence de discrimination indirecte
- **Variables auto-déclarées** (enquêtes) → biais de désirabilité sociale possible
- **Données transversales** (pas de suivi longitudinal) → relations observées ≠ causalité
